# FINM 32000: Homework 4

By Andrew McLaughlin

## Problem 1

Assume that the short rate (the instantaneous spot rate of interest) follows the process

$$
dr_t = \mu(r_t, t)\,dt + \sigma(r_t, t)\,dW_t
$$

where $W_t$ is Brownian motion under risk-neutral probabilities. This framework includes models such as the Vasicek and CIR models, which correspond to particular choices of the functions $(\mu, \sigma)$, but for part (a), let's leave $\mu$ and $\sigma$ as unspecified functions.

### (a)

Consider an interest rate derivative whose time-$T$ payout has value given by some function $F(r_T)$, and whose time-$t$ price $C_t$ satisfies $C_t = C(r_t, t)$ for some smooth pricing function $C$.

Apply Ito's rule to find the risk-neutral dynamics of $C$. Then set its drift equal to $rC$, to derive a PDE for $C(r, t)$.

---

Suppose, in particular, that the risk-neutral dynamics of $r$ are given by a Vasicek model

$$
dr_t = \kappa(\theta - r_t)\,dt + \sigma\,dW_t,
$$

with parameters $\kappa = 3$, $\theta = 0.05$, $\sigma = 0.03$. Consider a $T = 5$-year discount bond (a zero-coupon bond which pays 1 at maturity $T$).

Suppose, in particular, that the risk-neutral dynamics of $r$ are given by a Vasicek model

$$
dr_t = \kappa(\theta - r_t)\,dt + \sigma\,dW_t,
$$

with parameters $\kappa = 3$, $\theta = 0.05$, $\sigma = 0.03$. Consider a $T = 5$-year discount bond (a zero-coupon bond which pays 1 at maturity $T$).

### (b)

Write code to find the time-0 price of the bond by applying a standard central-difference explicit finite difference scheme to the PDE in (a). (Therefore $C_j^n$ will be determined by $C_{j+1}^{n+1}$, $C_j^{n+1}$, and $C_{j-1}^{n+1}$.)

Complete the code in the file `finm320-26-hw4.ipynb`.

In [1]:
import numpy as np

In [2]:
class Vasicek:

    def __init__(self,kappa,theta,sigma):
        self.kappa=kappa
        self.theta=theta
        self.sigma=sigma

In [3]:
hw41dynamics = Vasicek(kappa=3,theta=0.05,sigma=0.03)

In [4]:
class Bond:

    def __init__(self, T):
        self.T=T


In [5]:
hw41contract = Bond(T=5)

In [6]:
class FDexplicitEngine:

    def __init__(self, rMax, rMin, deltar, deltat, useUpwind):
        self.rMax=rMax
        self.rMin=rMin
        self.deltar=deltar
        self.deltat=deltat
        self.useUpwind=useUpwind

    def price_bond_vasicek(self,contract,dynamics):
    # You complete the coding of this function
    #
    # Returns array of all initial short rates,
    # and the corresponding array of zero-coupon
    # T-maturity bond prices

        T = contract.T
        N=round(T/self.deltat)
        if abs(N-T/self.deltat) > 1e-12:
            raise ValueError("Bad delta t")

        r=np.arange(self.rMax,self.rMin-self.deltar/2,-self.deltar)   #I'm making the FIRST indices of the array correspond to HIGH levels of r
        bondprice=np.ones(np.size(r))


        if self.useUpwind:
            mu  = dynamics.kappa * (dynamics.theta - r)
            a   = dynamics.sigma**2 * self.deltat / self.deltar**2
            qu  = 0.5 * a + np.maximum(mu, 0) * self.deltat / self.deltar
            qd  = 0.5 * a + np.maximum(-mu, 0) * self.deltat / self.deltar
            qm  = 1 - a - np.abs(mu) * self.deltat / self.deltar
        else:
            qu = 0.5 * (dynamics.sigma**2 * self.deltat / self.deltar**2 + dynamics.kappa * (dynamics.theta - r) * self.deltat / self.deltar)
            qd = 0.5 * (dynamics.sigma**2 * self.deltat / self.deltar**2 - dynamics.kappa * (dynamics.theta - r) * self.deltat / self.deltar)
            qm = (1 - dynamics.sigma**2 * self.deltat / self.deltar**2) * np.ones(np.size(r))



        for t in np.arange(N-1,-1,-1)*self.deltat:
            # Do not change any of the code in this loop

            bondprice[1:-1]=1/(1+r[1:-1]*self.deltat)*(qd[1:-1]*bondprice[2:]+qm[1:-1]*bondprice[1:-1]+qu[1:-1]*bondprice[:-2])
            # We are only calculating the interior grid points here, so
            # bondprice, r, qd, qm, and qu have [1:-1] indexes

            # For this contract, it is not obvious
            # what boundary conditions to use at the top and bottom,
            # so let us assume "linearity" boundary conditions
            bondprice[0]=2*bondprice[1]-bondprice[2]
            bondprice[-1]=2*bondprice[-2]-bondprice[-3]

        return (r, bondprice)

In [7]:
hw41FD = FDexplicitEngine(rMax=0.35,rMin=-0.25,deltar=0.01,deltat=0.01,useUpwind=False)

In [8]:
(r, bondprice) = hw41FD.price_bond_vasicek(hw41contract,hw41dynamics)

In [9]:
np.set_printoptions(precision=4,suppress=True)
displayrows=(r<0.15+hw41FD.deltar/2) & (r>0.0-hw41FD.deltar/2)

In [10]:
print(np.stack((r, bondprice),axis=1)[displayrows])

[[ 1.5000e-01 -1.4273e+09]
 [ 1.4000e-01  1.6361e+08]
 [ 1.3000e-01  2.2294e+07]
 [ 1.2000e-01 -1.3724e+06]
 [ 1.1000e-01 -1.3361e+05]
 [ 1.0000e-01  3.2966e+03]
 [ 9.0000e-02  1.3021e+02]
 [ 8.0000e-02  7.7128e-01]
 [ 7.0000e-02  7.7385e-01]
 [ 6.0000e-02  7.7643e-01]
 [ 5.0000e-02  7.7902e-01]
 [ 4.0000e-02  7.8162e-01]
 [ 3.0000e-02  7.8423e-01]
 [ 2.0000e-02  7.8685e-01]
 [ 1.0000e-02  1.4165e+03]
 [-3.3307e-16  5.1498e+04]]


### (c)

Also write code to price the bond using an explicit upwind approximation to $\frac{\partial C}{\partial r}$ instead of the usual central difference. Specifically, for those $r_j$ such that $\kappa(\theta - r_j) \geq 0$, approximate $\frac{\partial C}{\partial r}(r_j, t_{n+1})$ using the points $C_{j+1}^{n+1}$ and $C_j^{n+1}$. For those $r_j$ such that $\kappa(\theta - r_j) < 0$, approximate $\frac{\partial C}{\partial r}(r_j, t_{n+1})$ using the points $C_j^{n+1}$ and $C_{j-1}^{n+1}$. (For $\frac{\partial^2 C}{\partial r^2}$, use the usual central-difference approximation.)

In (b) and (c), to approximate the PDE's $rC$ term, use the values of $r$ and $C$ at node $(n, j)$. (As we said in class, node $(n+1, j)$ would also be a natural choice, but let's choose $n$ instead of $n+1$.) At the grid's upper and lower boundaries $r_{\max}$ and $r_{\min}$, impose for all $t < T$ the "linearity" boundary conditions:

$$
C(r_{\max}, t) = 2C(r_{\max} - \Delta r, t) - C(r_{\max} - 2\Delta r, t)
$$

$$
C(r_{\min}, t) = 2C(r_{\min} + \Delta r, t) - C(r_{\min} + 2\Delta r, t)
$$

(This technique can help in some situations where it is not obvious what boundary conditions to use.) Thus, in each column of the grid, first solve for $C$ in the interior nodes; then deal with the top and bottom nodes.

In [11]:
hw41FD = FDexplicitEngine(rMax=0.35,rMin=-0.25,deltar=0.01,deltat=0.01,useUpwind=True)
(r, bondprice) = hw41FD.price_bond_vasicek(hw41contract,hw41dynamics)
np.set_printoptions(precision=4,suppress=True)
displayrows=(r<0.15+hw41FD.deltar/2) & (r>0.0-hw41FD.deltar/2)
print(np.stack((r, bondprice),axis=1)[displayrows])

[[ 0.15    0.7536]
 [ 0.14    0.7561]
 [ 0.13    0.7586]
 [ 0.12    0.7611]
 [ 0.11    0.7637]
 [ 0.1     0.7662]
 [ 0.09    0.7688]
 [ 0.08    0.7713]
 [ 0.07    0.7739]
 [ 0.06    0.7765]
 [ 0.05    0.7791]
 [ 0.04    0.7817]
 [ 0.03    0.7843]
 [ 0.02    0.7869]
 [ 0.01    0.7895]
 [-0.      0.7922]]


### (d)

Suppose $f : \mathbb{R} \to \mathbb{R}$ is smooth in some open neighborhood of $x$. Show that as $h \to 0$,

$$
\frac{f(x+h) - f(x)}{h} - f'(x) = O(h)
\qquad \text{and} \qquad
\frac{f(x+h) - f(x-h)}{2h} - f'(x) = O(h^2)
$$

using Taylor's theorem. The $O(h)$ means "some function bounded by a constant times $h$, near $h = 0$." Likewise, $O(h^2)$ means "some function bounded by a constant times $h^2$, near $h = 0$." Different instances of "$O$" may mean different functions. The "constants" may depend on $x$ but not $h$.


### (e)

For all part (e) calculations: Use the grid spacings $\Delta r = 0.01$ and $\Delta t = 0.01$. Use $r_{\max} = 0.35$ and $r_{\min} = -0.25$ for the upper and lower boundaries of the grid, respectively.

Run a central-difference calculation and an upwind calculation of the bond price for $r_0 = 0.10$. Which is more accurate? The more accurate of the two solutions should agree, to three significant digits, with the exact bond price in this model: **0.7661**. The less accurate of the two solutions will be very inaccurate.

**Answer:** The upwind method is more accurate.

### (f)

Based on your answers to (d) and (e), insert either **"greater"** or **"less"** in each blank space in the following rule-of-thumb. No explanation necessary.

> Ignoring stability issues and considering only consistency (i.e. "truncation error," also known as "local discretization error"), the upwind explicit scheme, which uses one-sided spatial differences, discretizes the PDE with ________ accuracy than the standard explicit scheme, which uses central spatial differences.
>
> However, to actually guarantee convergence, the grid spacing must satisfy certain stability constraints, to prevent errors from propagating explosively. In a PDE exhibiting strong drift, we have seen that these constraints may allow the upwind scheme ________ freedom in choosing grid spacing, compared to the central scheme.

**Answer:** less, greater


### (g)

The continuously-compounded yield-to-maturity of a zero-coupon bond with time-$t$ price $P_t$ and nonrandom face value $P_T$ to be paid at maturity date $T$ is

$$
\frac{\log(P_T / P_t)}{T - t}
$$

where, as always for us, $\log$ denotes natural log, and where $P_T = 1$ according to this problem's assumptions. One way to think of the time-$t$ yield to maturity $T$ is as the average of some type of time-$t$ expectation of the instantaneous spot rates from time $t$ to time $T$.

Find the yield-to-maturity of a 5-year discount bond, in the case that $r_0 = 0.12$, and in the case that $r_0 = 0.02$. (The "good" results from part (e) may be used here. The "bad" results should not be used, unless you want to fix them by modifying the grid spacings.)

Why, intuitively, is the yield for $r_0 = 0.12$ smaller than $0.12$, whereas the yield for $r_0 = 0.02$ is greater than $0.02$?

> **Comment:** Under these short-rate dynamics, there do exist analytic pricing formulas for bonds. So we do not need finite difference methods to value the simple payoff that we have here. But the finite difference scheme can be modified to handle contracts for which exact pricing formulas do not exist.


In [14]:
ytm = -np.log(bondprice) / hw41contract.T
print(np.stack((r, bondprice, ytm), axis=1)[displayrows])


[[ 0.15    0.7536  0.0566]
 [ 0.14    0.7561  0.0559]
 [ 0.13    0.7586  0.0553]
 [ 0.12    0.7611  0.0546]
 [ 0.11    0.7637  0.0539]
 [ 0.1     0.7662  0.0533]
 [ 0.09    0.7688  0.0526]
 [ 0.08    0.7713  0.0519]
 [ 0.07    0.7739  0.0513]
 [ 0.06    0.7765  0.0506]
 [ 0.05    0.7791  0.0499]
 [ 0.04    0.7817  0.0493]
 [ 0.03    0.7843  0.0486]
 [ 0.02    0.7869  0.0479]
 [ 0.01    0.7895  0.0473]
 [-0.      0.7922  0.0466]]


**Answer:** Under the Vasicek mdoel, the yield to maturity (YTM) will revert to $\theta = 0.05$. If $\theta$ is the long run mean of the short rate for the bond, then $r_t$ will move towards $0.05$ over time. If $r_0$ is$0.12$, the the YTM is expected to fall towards $0.05$, so the averaage rate over 5 years is expected to be less than $0.12$. If $r_0$ is$0.02$, the the YTM is expected to rise towards $0.05$, so the averaage rate over 5 years is expected to be greater than $0.02$. 